# Nosso primeiro agente com LangChain

### Sistemas Multiagentes / Agentic AI

Neste notebook vamos construir **o mesmo agente em etapas**.

> **O que precisamos acrescentar para nosso agente fazer mais uma coisa?**

```text
agente mínimo
→ primeira ferramenta
→ duas ferramentas
→ tool loop
→ estado da execução
→ memória de curto prazo
→ conversa livre
```

Em vários momentos aparecerá **ANTES DE EXECUTAR**. Pare e faça uma previsão antes de rodar a célula.

## 0. Preparação

Vamos usar:

- **LangChain** para criar o agente;
- **Gemini** como modelo;
- funções Python comuns como ferramentas;
- uma lista Python como nosso “sistema de chamados” fictício.

### Chave do Gemini no Colab

No Colab, abra o painel **Secrets** (ícone de chave) e crie:

`GEMINI_API_KEY`

Permita que o notebook acesse o segredo.

> Não coloque sua chave diretamente no código.

In [ ]:
%pip install -q -U langchain langchain-google-genai

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("Chave configurada para esta sessão.")

In [ ]:
from langchain.agents import create_agent

MODELO = "google_genai:gemini-3.7-flash"

print("Modelo:", MODELO)

A API atual do LangChain permite passar o modelo diretamente a `create_agent()` usando o formato `provider:model`.

```python
model="google_genai:gemini-3.7-flash"
```

Para este primeiro contato, isso é mais simples do que criar manualmente um objeto `ChatGoogleGenerativeAI`. Se um dia precisarmos configurar detalhes específicos do modelo, podemos usar o objeto explicitamente.

# 1. O menor agente que podemos criar

Começaremos sem ferramentas.

```text
AGENTE
└── modelo
```

Ele recebe mensagens e produz respostas.

Isso é propositalmente limitado.

> **Pergunta:** um sistema que apenas responde em linguagem natural consegue produzir uma consequência no nosso sistema de chamados?

In [ ]:
INSTRUCOES = """
Você é o CampusBot, um agente didático de um campus.
Seja breve e não afirme que executou uma ação externa se nenhuma ferramenta permitiu executá-la.
"""

agente_basico = create_agent(
    model=MODELO,
    tools=[],
    system_prompt=INSTRUCOES,
)

resultado = agente_basico.invoke({
    "messages": [
        {
            "role": "user",
            "content": "O projetor do Lab 4 não funciona. Abra um chamado."
        }
    ]
})

print(resultado["messages"][-1].text)

### Pare e interprete

O framework criou um agente, mas demos a ele apenas um modelo.

Perguntas:

1. Ele conseguiu produzir uma resposta?
2. Algum sistema externo foi alterado?
3. Se ele disser “abri o chamado”, isso prova que um chamado foi realmente criado?

> **Responder não é agir.**

Vamos dar ao agente sua primeira capacidade externa.

# 2. Nosso ambiente

Antes da ferramenta, precisamos de alguma coisa que possa ser consultada ou modificada.

Nesta prática, nosso “mundo externo” será apenas uma lista Python.

Não há banco, HTTP ou API real porque eles não acrescentariam nada à ideia que queremos observar.

In [ ]:
CHAMADOS = []
SERVICO_DISPONIVEL = True


def resetar_ambiente(*, chamado_no_lab4=False):
    """Volta o ambiente a um estado conhecido para repetir experimentos."""
    global SERVICO_DISPONIVEL

    CHAMADOS.clear()
    CHAMADOS.append({
        "protocolo": "CH-001",
        "local": "Lab 2",
        "problema": "ar-condicionado não liga",
        "status": "aberto",
    })

    if chamado_no_lab4:
        CHAMADOS.append({
            "protocolo": "CH-002",
            "local": "Lab 4",
            "problema": "projetor não funciona",
            "status": "aberto",
        })

    SERVICO_DISPONIVEL = True


resetar_ambiente()
CHAMADOS

# 3. Primeira ferramenta: abrir um chamado

Primeiro escreveremos apenas uma função Python.

Ela sabe fazer algo que a LLM, sozinha, não consegue: **alterar nosso ambiente**.

Para este caso simples, o LangChain aceita uma função Python diretamente como tool. Não precisamos de um decorador adicional.

In [ ]:
def abrir_chamado(local: str, problema: str) -> dict:
    """Abre um novo chamado de suporte para um problema em um local do campus."""
    if not SERVICO_DISPONIVEL:
        return {
            "ok": False,
            "erro": "serviço de chamados indisponível",
        }

    protocolo = f"CH-{len(CHAMADOS) + 1:03d}"
    chamado = {
        "protocolo": protocolo,
        "local": local,
        "problema": problema,
        "status": "aberto",
    }
    CHAMADOS.append(chamado)

    return {
        "ok": True,
        "chamado": chamado,
    }

In [ ]:
import inspect

print("Nome:", abrir_chamado.__name__)
print("Descrição:", abrir_chamado.__doc__)
print("Assinatura:", inspect.signature(abrir_chamado))

### O que o LangChain consegue enxergar?

Ao receber essa função na lista `tools`, o framework pode usar principalmente:

```text
nome da função
+ docstring
+ tipos dos argumentos
```

para apresentar a capacidade ao modelo.

> **ANTES DE EXECUTAR:** se o usuário pedir para abrir um chamado, qual será a sequência esperada?

```text
usuário
→ ?
→ ?
→ resposta
```

In [ ]:
def mostrar_trajetoria(resultado):
    """Mostra apenas os eventos mais importantes da execução."""
    print("=" * 72)

    for mensagem in resultado["messages"]:
        if mensagem.type == "human":
            print("\n👤 USUÁRIO")
            print(mensagem.content)

        elif mensagem.type == "ai":
            for chamada in (mensagem.tool_calls or []):
                print("\n🤖 MODELO SOLICITOU UMA FERRAMENTA")
                print("ferramenta:", chamada["name"])
                print("argumentos:", chamada["args"])

            if mensagem.text:
                print("\n🤖 MODELO RESPONDEU")
                print(mensagem.text)

        elif mensagem.type == "tool":
            print("\n🔧 RESULTADO DA FERRAMENTA")
            print("ferramenta:", mensagem.name)
            print(mensagem.content)

    print("\n" + "=" * 72)

In [ ]:
resetar_ambiente()

agente_com_uma_tool = create_agent(
    model=MODELO,
    tools=[abrir_chamado],
    system_prompt=INSTRUCOES,
)

print("AMBIENTE ANTES:")
print(CHAMADOS)

resultado = agente_com_uma_tool.invoke({
    "messages": [
        {
            "role": "user",
            "content": "O projetor do Lab 4 não funciona. Abra um chamado."
        }
    ]
})

mostrar_trajetoria(resultado)

print("\nAMBIENTE DEPOIS:")
print(CHAMADOS)

### Pergunta central

**Quem abriu o chamado?**

Não foi a LLM.

A LLM **solicitou** uma ferramenta.  
O runtime do agente executou a função Python.  
A função modificou o ambiente.  
O resultado voltou para o modelo.

```text
modelo solicita
      ↓
programa executa
      ↓
ferramenta altera o ambiente
      ↓
resultado volta ao agente
```

Agora nosso agente consegue fazer uma coisa. Vamos dar a ele uma segunda capacidade.

# 4. Segunda ferramenta: consultar antes de agir

Novo requisito:

> “Veja se já existe um chamado para o projetor do Lab 4. Se não existir, abra um.”

Agora uma única ação pode não ser suficiente.

Precisamos permitir que o agente **descubra alguma coisa antes de decidir o que fazer depois**.

In [ ]:
def consultar_chamados(local: str) -> list[dict]:
    """Consulta os chamados existentes em um local do campus."""
    return [
        chamado.copy()
        for chamado in CHAMADOS
        if chamado["local"].lower() == local.lower()
    ]

> **ANTES DE EXECUTAR**

O ambiente começa sem chamado no Lab 4.

Faça uma previsão:

1. Qual ferramenta deve ser usada primeiro?
2. O que ela deve devolver?
3. O agente precisa usar outra ferramenta?
4. Quando o agente deve parar?

In [ ]:
resetar_ambiente()

agente = create_agent(
    model=MODELO,
    tools=[consultar_chamados, abrir_chamado],
    system_prompt=INSTRUCOES,
)

pedido = """
O projetor do Lab 4 não funciona.
Veja se já existe um chamado nesse local.
Se não existir, abra um chamado para o problema.
"""

resultado = agente.invoke({
    "messages": [{"role": "user", "content": pedido}]
})

mostrar_trajetoria(resultado)

print("\nAMBIENTE FINAL:")
print(CHAMADOS)

## Onde foi parar o nosso loop?

Observe a trajetória que acabou de ocorrer:

```text
modelo
→ consultar_chamados
→ resultado da consulta
→ modelo novamente
→ abrir_chamado
→ resultado da abertura
→ modelo novamente
→ resposta final
```

Nós **não escrevemos**:

```python
while ...
    decidir()
    executar_ferramenta()
    observar_resultado()
```

Mas o loop continua existindo.

`create_agent()` passou a executar essa mecânica por nós.

> **Framework não elimina a arquitetura. Ele encapsula parte dela.**

# 5. Onde está o estado dessa execução?

O agente precisou usar o resultado da primeira ferramenta para decidir o próximo passo.

No LangChain, o estado básico do agente contém uma sequência de mensagens.

Vamos olhar para ela sem os detalhes visuais.

In [ ]:
for i, mensagem in enumerate(resultado["messages"], start=1):
    print(
        f"{i:02d}",
        f"tipo={mensagem.type:<5}",
        f"classe={mensagem.__class__.__name__}"
    )

Observe que aparecem mensagens do usuário, do modelo e das ferramentas.

O resultado da ferramenta não “desapareceu”: ele entrou na trajetória disponível para a decisão seguinte.

Neste notebook não vamos criar um `AgentState` customizado. Primeiro queremos reconhecer o estado que o framework já mantém.

---

## Experimento A — o ambiente já contém o chamado

Agora o mesmo pedido será feito, mas o ambiente começará com um chamado existente no Lab 4.

> **ANTES DE EXECUTAR:** o agente deveria abrir outro chamado?

In [ ]:
resetar_ambiente(chamado_no_lab4=True)

resultado = agente.invoke({
    "messages": [{"role": "user", "content": pedido}]
})

mostrar_trajetoria(resultado)

print("\nAMBIENTE FINAL:")
print(CHAMADOS)

### O que mudou?

- modelo: o mesmo;
- instruções: as mesmas;
- ferramentas: as mesmas;
- pedido: o mesmo;
- **estado observado do ambiente: diferente**.

O comportamento pode mudar porque o agente recebeu uma percepção diferente.

## Experimento B — retire uma capacidade

Criaremos agora um agente que **só consegue consultar**.

Depois pediremos novamente para abrir um chamado.

> O modelo pode executar uma capacidade que não foi disponibilizada como ferramenta?

In [ ]:
agente_so_consulta = create_agent(
    model=MODELO,
    tools=[consultar_chamados],
    system_prompt=INSTRUCOES,
)

resultado = agente_so_consulta.invoke({
    "messages": [{
        "role": "user",
        "content": "Abra um chamado para o projetor do Lab 4."
    }]
})

mostrar_trajetoria(resultado)

A LLM pode **falar sobre** abrir um chamado.

Mas, neste harness, não existe uma ferramenta que permita produzir esse efeito.

As capacidades disponíveis fazem parte da arquitetura do agente.

## Experimento C — a ferramenta pode falhar

Agora a ferramenta existe, mas o serviço ficará indisponível.

> **ANTES DE EXECUTAR:** qual seria uma resposta correta do agente?

Compare estas duas possibilidades:

- “Chamado aberto com sucesso.”
- “Não consegui abrir o chamado porque o serviço está indisponível.”

Qual delas é sustentada pela evidência?

In [ ]:
resetar_ambiente()

SERVICO_DISPONIVEL = False

resultado = agente.invoke({
    "messages": [{
        "role": "user",
        "content": "Abra um chamado para o projetor do Lab 4."
    }]
})

mostrar_trajetoria(resultado)

SERVICO_DISPONIVEL = True

### Três eventos diferentes

```text
solicitar uma ferramenta
≠
executar a ferramenta
≠
a ferramenta ter sucesso
```

Esse detalhe será importante quando estudarmos diagnóstico, avaliação e confiabilidade de agentes.

# 6. Memória de curto prazo: continuar a mesma conversa

Até aqui, cada `invoke()` começou uma nova conversa.

Vamos fazer duas interações:

1. abrir um chamado;
2. perguntar depois: “Qual foi mesmo o protocolo?”

Para isso, adicionaremos um **checkpointer** e identificaremos a conversa com um `thread_id`.

> Aqui estamos usando memória **de curto prazo da thread**.  
> `InMemorySaver` guarda os dados apenas enquanto este runtime Python existir. Reiniciar o Colab apaga essa memória.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

resetar_ambiente()

agente_com_memoria = create_agent(
    model=MODELO,
    tools=[consultar_chamados, abrir_chamado],
    system_prompt=INSTRUCOES,
    checkpointer=InMemorySaver(),
)

config_turma = {
    "configurable": {
        "thread_id": "grupo-01"
    }
}

In [ ]:
# Primeira interação da mesma conversa
resultado_1 = agente_com_memoria.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Abra um chamado para o projetor do Lab 4."
        }]
    },
    config=config_turma,
)

mostrar_trajetoria(resultado_1)

> **ANTES DE EXECUTAR:** sem repetir o protocolo na nova pergunta, o agente conseguirá recuperá-lo?

In [ ]:
# Segunda interação da MESMA conversa: mesmo thread_id
resultado_2 = agente_com_memoria.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Qual foi mesmo o protocolo que você acabou de abrir?"
        }]
    },
    config=config_turma,
)

print(resultado_2["messages"][-1].text)

## Mesma aplicação, outra thread

Agora faremos exatamente a mesma pergunta usando outro `thread_id`.

> **ANTES DE EXECUTAR:** essa nova conversa deveria conhecer o protocolo anterior?

In [ ]:
outra_conversa = {
    "configurable": {
        "thread_id": "grupo-02"
    }
}

resultado_3 = agente_com_memoria.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Qual foi mesmo o protocolo que você acabou de abrir?"
        }]
    },
    config=outra_conversa,
)

print(resultado_3["messages"][-1].text)

### O que o `thread_id` nos mostrou?

```text
mesmo agente
+ mesma aplicação
+ thread diferente
→ histórico diferente
```

A memória não é uma propriedade mágica da LLM.

O harness precisa decidir:

- o que guardar;
- onde guardar;
- a qual conversa aquela informação pertence.

Neste exemplo o LangChain/LangGraph assumiu parte dessa responsabilidade por meio do checkpointer.

# Converse livremente com o agente

Agora vamos usar o mesmo agente como um pequeno chat.

Ele possui:

```text
modelo
+ consultar_chamados
+ abrir_chamado
+ memória da thread
```

Experimente, por exemplo:

- `Quais chamados existem no Lab 2?`
- `Abra um chamado para o Wi-Fi do Lab 5.`
- `Qual protocolo você acabou de abrir?`
- `Veja se já existe chamado no Lab 4; se não existir, abra um para o projetor.`

Digite `sair` para encerrar.

O chat reutiliza a mesma `thread_id`, então as mensagens anteriores ficam disponíveis enquanto o runtime do Colab estiver ativo.

In [ ]:
chat_config = {
    "configurable": {
        "thread_id": "chat-livre"
    }
}

print("CampusBot iniciado. Digite 'sair' para encerrar.\n")

while True:
    mensagem = input("Você: ").strip()

    if mensagem.lower() in {"sair", "exit", "quit"}:
        print("CampusBot: Até mais!")
        break

    if not mensagem:
        continue

    resultado = agente_com_memoria.invoke(
        {
            "messages": [{
                "role": "user",
                "content": mensagem,
            }]
        },
        config=chat_config,
    )

    print("CampusBot:", resultado["messages"][-1].text)
    print()

---

## O que construímos

```text
AGENTE
├── modelo
├── instruções
├── ferramentas
│   ├── consultar_chamados
│   └── abrir_chamado
├── tool loop            ← conduzido pelo framework
├── estado               ← messages
└── memória da conversa  ← checkpointer + thread_id
```

O framework reduziu o código necessário. As responsabilidades arquiteturais continuam existindo.

### Referências técnicas

- Agents: https://docs.langchain.com/oss/python/langchain/agents
- Google Generative AI: https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai
- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory